In [13]:
# 03_image_extraction.ipynb

# --- CELL 1: Setup ---
from __future__ import annotations
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm

DATA_DIR = Path("./data/raw")
PATIENT_DIR = DATA_DIR / "AngioCAD_Dataset"
OUTPUT_DIR = Path("./data/processed")
FRAMES_DIR = OUTPUT_DIR / "frames"
FRAMES_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = (224, 224)
MAX_FRAMES_PER_SERIES = 10

print(f"Frames directory: {FRAMES_DIR}")

Frames directory: data\processed\frames


In [14]:
# --- CELL 2: Helper Functions ---
def list_series_files(series_path: Path) -> list[Path]:
    """List all image files in a series directory."""
    files: list[Path] = []
    for pattern in ("*.jpg", "*.jpeg", "*.png"):
        files.extend(sorted(series_path.glob(pattern)))
    return files


def sample_frames(files: list[Path], k: int) -> list[Path]:
    """Evenly sample k frames from a list of files."""
    if len(files) <= k:
        return files
    idx = np.linspace(0, len(files) - 1, k).astype(int)
    return [files[i] for i in idx]


def load_gray(path: Path, size: tuple[int, int]) -> np.ndarray | None:
    """Load an image as grayscale and resize it."""
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    return cv2.resize(img, size, interpolation=cv2.INTER_AREA)


def split_series(value) -> list[str]:
    """Split a series string like '1-3-5' into ['1', '3', '5']."""
    if pd.isna(value):
        return []
    return [token.strip() for token in str(value).split("-") if token.strip()]

In [15]:
# --- CELL 3: Load Source Table ---
df = pd.read_csv(OUTPUT_DIR / "clinical_features_original.csv")

RIGHT_COL = "Right Coronary Series_y" if "Right Coronary Series_y" in df.columns else "Right Coronary Series"
LEFT_COL = "Left Coronary Series_y" if "Left Coronary Series_y" in df.columns else "Left Coronary Series"

print(f"Right column: {RIGHT_COL}")
print(f"Left  column: {LEFT_COL}")
print(f"Patients    : {len(df)}")

Right column: Right Coronary Series_y
Left  column: Left Coronary Series_y
Patients    : 377


In [16]:
# --- CELL 4: Extract Frames ---
records: list[dict] = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting"):
    pid = str(int(row["ID"]))
    patient_dir = PATIENT_DIR / pid
    if not patient_dir.exists():
        continue

    for side, col in (("R", RIGHT_COL), ("L", LEFT_COL)):
        for series_id in split_series(row.get(col)):
            series_dir = patient_dir / series_id
            if not series_dir.exists():
                continue

            files = list_series_files(series_dir)
            chosen = sample_frames(files, MAX_FRAMES_PER_SERIES)

            for idx, path in enumerate(chosen):
                img = load_gray(path, IMG_SIZE)
                if img is None:
                    continue
                out_name = f"{pid}_{side}_{series_id}_{idx:02d}.png"
                out_path = FRAMES_DIR / out_name
                cv2.imwrite(str(out_path), img)
                records.append({
                    "ID": pid,
                    "side": side,
                    "series": series_id,
                    "frame_idx": idx,
                    "path": str(out_path),
                })
                
print(f"Extracted {len(records)} frames")

Extracting: 100%|██████████| 377/377 [00:51<00:00,  7.37it/s]

Extracted 13353 frames


In [17]:
# --- CELL 5: Persist Frame Index ---
frames_df = pd.DataFrame(records)
frames_df.to_csv(OUTPUT_DIR / "frames_index.csv", index=False)

print(f"Patients with frames: {frames_df['ID'].nunique()}")
print(f"Total frames        : {len(frames_df)}")
print(frames_df["side"].value_counts())

Patients with frames: 376
Total frames        : 13353
side
L    7137
R    6216
Name: count, dtype: int64
